In [1]:
import pandas as pd
import re

In [2]:
input_file_name_list = [ "UD_Italian-ISDT/it_isdt-ud-train.conllu" , 
                        "UD_Italian-ISDT/it_isdt-ud-test.conllu" , 
                        "UD_Italian-ISDT/it_isdt-ud-dev.conllu" , 
                        "UD_Italian-MarkIT/it_markit-ud-train.conllu" , 
                        "UD_Italian-MarkIT/it_markit-ud-test.conllu" , 
                        "UD_Italian-MarkIT/it_markit-ud-dev.conllu" ,
                        "UD_Italian-ParlaMint/it_parlamint-ud-train.conllu" , 
                        "UD_Italian-ParlaMint/it_parlamint-ud-test.conllu" , 
                        "UD_Italian-POSTWITA/it_postwita-ud-train.conllu" , 
                        "UD_Italian-POSTWITA/it_postwita-ud-test.conllu" , 
                        "UD_Italian-POSTWITA/it_postwita-ud-dev.conllu" ,
                        "UD_Italian-TWITTIRO/it_twittiro-ud-train.conllu" , 
                        "UD_Italian-TWITTIRO/it_twittiro-ud-test.conllu" , 
                        "UD_Italian-TWITTIRO/it_twittiro-ud-dev.conllu" ,
                        "UD_Italian-VIT/it_vit-ud-train.conllu" ,
                        "UD_Italian-VIT/it_vit-ud-test.conllu" ,
                        "UD_Italian-VIT/it_vit-ud-dev.conllu"
                       ]

In [3]:
output_file_name_list = list()

In [4]:
def convert_to_csv(input_file_name):
        
    input_file = open(input_file_name, encoding="UTF8")
    
    output_file_name = re.sub("conllu","csv",input_file_name)
    
    output_file = open(output_file_name, "w", encoding="UTF8")

    for line in input_file:
        if re.match("^[0-9].+",line):
            line = re.sub("\"\t\.\tPUNCT","\"\t\"\tPUNCT",line)# needed to handle mistakes in lemmatization of " in vit that make the .csv unparsable by pandas
            output_file.write(line)
    
    output_file_name_list.append(output_file_name)

<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\.'
C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_19064\3727954920.py:11: SyntaxWarning: invalid escape sequence '\.'
  line = re.sub("\"\t\.\tPUNCT","\"\t\"\tPUNCT",line)# needed to handle mistakes in lemmatization of " in vit that make the .csv unparsable by pandas


In [5]:
for input_file_name in input_file_name_list:
    convert_to_csv(input_file_name)

In [6]:
df = pd.DataFrame(columns = ["ID", "FORM", "LEMMA", "UPOS", "XPOS", "FEATS", "HEAD", "DEPREL", "DEPS", "MISC"])

In [7]:
for output_file_name in output_file_name_list:
    print(output_file_name)
    partial_df = pd.read_csv(output_file_name, sep="\t", names = ["ID", "FORM", "LEMMA", "UPOS", "XPOS", "FEATS", "HEAD", "DEPREL", "DEPS", "MISC"] )
    df = pd.concat([df,partial_df])

UD_Italian-ISDT/it_isdt-ud-train.csv
UD_Italian-ISDT/it_isdt-ud-test.csv
UD_Italian-ISDT/it_isdt-ud-dev.csv
UD_Italian-MarkIT/it_markit-ud-train.csv
UD_Italian-MarkIT/it_markit-ud-test.csv
UD_Italian-MarkIT/it_markit-ud-dev.csv
UD_Italian-ParlaMint/it_parlamint-ud-train.csv
UD_Italian-ParlaMint/it_parlamint-ud-test.csv
UD_Italian-POSTWITA/it_postwita-ud-train.csv
UD_Italian-POSTWITA/it_postwita-ud-test.csv
UD_Italian-POSTWITA/it_postwita-ud-dev.csv
UD_Italian-TWITTIRO/it_twittiro-ud-train.csv
UD_Italian-TWITTIRO/it_twittiro-ud-test.csv
UD_Italian-TWITTIRO/it_twittiro-ud-dev.csv
UD_Italian-VIT/it_vit-ud-train.csv
UD_Italian-VIT/it_vit-ud-test.csv
UD_Italian-VIT/it_vit-ud-dev.csv


In [8]:
feats_frequencies_treebanks = df[df["UPOS"] == "VERB"]["FEATS"].value_counts().to_frame()

In [9]:
mapping = pd.read_csv("UD_paralex_mapping.csv", sep="\t", index_col="FEATS")

In [10]:
def map_to_cell(feats):
    
    if feats in mapping.index:
        cell = mapping.loc[feats,"cell"]
    
    else:
        cell = "?"
    
    return cell

In [11]:
df["cell"] = df["FEATS"].map(map_to_cell)

In [12]:
cell_frequencies_treebanks = df[df["UPOS"] == "VERB"]["cell"].value_counts().to_frame()

In [13]:
leffi_cells = pd.read_csv("../LeFFI_cells.csv", index_col="cell_id")

In [14]:
for i in leffi_cells.index:
    if i in cell_frequencies_treebanks.index:
        leffi_cells.loc[i,"frequency"] = cell_frequencies_treebanks.loc[i,"count"]

In [15]:
leffi_cells.to_csv("../LeFFI_cells.csv")